# Догрузка месяцев в таблицы MCC и типов карт (DRP)

Листы дашборда «MCC-коды» и «Терминалы» читают две отдельные таблицы, которые DAG не пополняет:

| Таблица | Зерно | Раньше грузила |
|---|---|---|
| `sbx_da.tmp_shestopalov_acq_mcc_month` | месяц × MCC | `mcc_month_upload_jan_aug.ipynb` |
| `sbx_da.tmp_shestopalov_acq_card_type_month` | месяц × MCC × платёжная система | `qc_card_type_vs_mcc.ipynb` + `card_type_month_upload_jan_aug.ipynb` |

Тетрадка считает в Impala только месяцы `MONTHS` теми же запросами, что и старые тетрадки, и в DRP **удаляет только эти месяцы и дописывает их**. Остальные месяцы и сами таблицы не трогает (без DROP).

Как запускать: Kernel → Restart Kernel and Run All Cells. Спросит логин keytab (Impala) и логин/пароль DRP. Если LDAP DRP отказал, не повторяйте больше 2 раз.

Каждый месяц — тяжёлый запрос к `scd1_trx` (несколько минут на таблицу). Посчитанные закрытые месяцы сохраняются в `Data/mcc_card_type_append/`, повторный запуск их не читает из Impala заново. Текущий (незакрытый) месяц пересчитывается всегда.

В следующий раз: поменять `MONTHS` (например, `['2026-11', '2026-12']`) и `OPEN_MONTH`.

In [ ]:
import getpass
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 100)

MONTHS = ['2026-09', '2026-10', '2026-11']
OPEN_MONTH = '2026-11'  # незакрытый месяц: всегда считать заново

DRP_SCHEMA = 'sbx_da'
MCC_TABLE = 'tmp_shestopalov_acq_mcc_month'
CARD_TABLE = 'tmp_shestopalov_acq_card_type_month'

MEM_LIMIT_FALLBACKS = ['8g', '4g']
FETCH_RETRIES = 3
IMPALA_USER_DEFAULT = 'Shestopalov-VYur'

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd()
CKPT_DIR = DATA_DIR / 'mcc_card_type_append'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert MONTHS == sorted(MONTHS) and all(re.fullmatch(r'\d{4}-\d{2}', m) for m in MONTHS)
assert min(MONTHS) >= '2026-09', 'Январь–август уже в таблицах, их не перегружаем'


def month_bounds(label):
    start = pd.Timestamp(label + '-01')
    end = start + pd.offsets.MonthBegin(1)
    return start.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d')


print('MONTHS =', MONTHS, '| OPEN_MONTH =', OPEN_MONTH)
print('checkpoints =', CKPT_DIR)

if 'imp' not in globals() or imp is None:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': input('Login (keytab): ').strip() or IMPALA_USER_DEFAULT},
    )
    imp._init_connection()
print('Impala: OK')

## 1) Запросы (как в старых тетрадках) и загрузка с повтором

In [ ]:
TRX_FILTER_CTE = '''
fiid_rshb AS (
  SELECT DISTINCT CAST(fa.c_fiid AS STRING) AS c_fiid
  FROM ods_alpha.scd1_base24_fiids fa
  WHERE COALESCE(CAST(fa.c_fiid_grp AS STRING), 'UNKNOWN') = 'RSHB'
),
sa_agr AS (
  SELECT DISTINCT CAST(a.n_agr AS STRING) AS n_agr
  FROM ods_alpha.scd1_agreements a
  WHERE UPPER(TRIM(CAST(a.acq_class AS STRING))) = 'SA'
    AND a.abs_agr_id IS NOT NULL
)'''


def sql_mcc_month(month_start, month_end_exclusive):
    # mcc_month_upload_jan_aug.ipynb, _sql_mcc_only_month
    return f'''
WITH {TRX_FILTER_CTE},
trx_base_raw AS (
  SELECT CAST(t.n_trx AS STRING) AS n_trx, CAST(t.n_mcc AS STRING) AS mcc, CAST(t.n_amt_src AS DOUBLE) AS n_amt_src
  FROM ods_alpha.scd1_trx t
  JOIN fiid_rshb fr ON fr.c_fiid = CAST(t.c_fiid_acq AS STRING)
  WHERE CAST(t.d_trx_orig AS TIMESTAMP) >= CAST('{month_start}' AS TIMESTAMP)
    AND CAST(t.d_trx_orig AS TIMESTAMP) < CAST('{month_end_exclusive}' AS TIMESTAMP)
    AND t.c_nter IS NOT NULL
    AND COALESCE(t.ods_deleted_flg, '0') <> '1'
    AND t.c_trx_class = 'SA'
    AND t.c_trx_type = 'S01'
    AND COALESCE(t.cf_trx_stat, '') <> 'R'
    AND t.n_mcc IS NOT NULL
),
trx_base AS (
  SELECT n_trx, MAX(mcc) AS mcc, MAX(n_amt_src) AS n_amt_src FROM trx_base_raw GROUP BY n_trx
),
ta AS (
  SELECT CAST(a.n_trx AS STRING) AS n_trx, CAST(a.n_agr AS STRING) AS n_agr,
         MAX(COALESCE(CAST(a.n_amt_tax AS DOUBLE), 0.0)) AS n_amt_tax
  FROM ods_alpha.scd1_trx_acq a
  JOIN trx_base tb ON tb.n_trx = CAST(a.n_trx AS STRING)
  JOIN sa_agr ss ON ss.n_agr = CAST(a.n_agr AS STRING)
  GROUP BY CAST(a.n_trx AS STRING), CAST(a.n_agr AS STRING)
)
SELECT tb.mcc, COUNT(DISTINCT tb.n_trx) AS trx_cnt, SUM(tb.n_amt_src) AS trx_sum, SUM(ta.n_amt_tax) AS commission_from_ops
FROM trx_base tb
JOIN ta ON ta.n_trx = tb.n_trx
GROUP BY tb.mcc
'''


def sql_card_type_month(month_start, month_end_exclusive):
    # qc_card_type_vs_mcc.ipynb, sql_month
    return f'''
WITH {TRX_FILTER_CTE},
trx_base_raw AS (
  SELECT CAST(t.n_trx AS STRING) AS n_trx, CAST(t.n_mcc AS STRING) AS mcc,
         CAST(t.c_fiid_iss AS STRING) AS c_fiid_iss, CAST(t.n_amt_src AS DOUBLE) AS n_amt_src
  FROM ods_alpha.scd1_trx t
  JOIN fiid_rshb fr ON fr.c_fiid = CAST(t.c_fiid_acq AS STRING)
  WHERE CAST(t.d_trx_orig AS TIMESTAMP) >= CAST('{month_start}' AS TIMESTAMP)
    AND CAST(t.d_trx_orig AS TIMESTAMP) < CAST('{month_end_exclusive}' AS TIMESTAMP)
    AND t.c_nter IS NOT NULL
    AND COALESCE(t.ods_deleted_flg, '0') <> '1'
    AND t.c_trx_class = 'SA'
    AND t.c_trx_type = 'S01'
    AND COALESCE(t.cf_trx_stat, '') <> 'R'
),
trx_base AS (
  SELECT n_trx, MAX(mcc) AS mcc, MAX(c_fiid_iss) AS c_fiid_iss, MAX(n_amt_src) AS n_amt_src
  FROM trx_base_raw GROUP BY n_trx
),
ta AS (
  SELECT CAST(a.n_trx AS STRING) AS n_trx, MAX(COALESCE(CAST(a.n_amt_tax AS DOUBLE), 0.0)) AS n_amt_tax
  FROM ods_alpha.scd1_trx_acq a
  JOIN trx_base tb ON tb.n_trx = CAST(a.n_trx AS STRING)
  JOIN sa_agr ss ON ss.n_agr = CAST(a.n_agr AS STRING)
  GROUP BY CAST(a.n_trx AS STRING)
)
SELECT
  COALESCE(NULLIF(TRIM(CAST(fi.c_fiid_desc AS STRING)), ''), 'empty') AS card_type,
  COALESCE(NULLIF(TRIM(tb.mcc), ''), 'empty') AS mcc,
  COUNT(DISTINCT tb.n_trx) AS trx_cnt,
  SUM(tb.n_amt_src) AS trx_sum,
  SUM(ta.n_amt_tax) AS commission_from_ops
FROM trx_base tb
JOIN ta ON ta.n_trx = tb.n_trx
LEFT JOIN ods_alpha.scd1_base24_fiids fi ON CAST(fi.c_fiid AS STRING) = tb.c_fiid_iss
GROUP BY 1, 2
'''


def _ckpt(kind, label):
    return CKPT_DIR / f'{kind}_{label}.csv.gz'


def _is_admission_fail(exc):
    s = str(exc).lower()
    return any(x in s for x in ('not enough memory', 'admission', 'queued reason', 'exceeded timeout', 's00051'))


def fetch_months(kind, sql_builder):
    parts = []
    for label in MONTHS:
        path = _ckpt(kind, label)
        if path.exists() and label != OPEN_MONTH:
            part = pd.read_csv(path, compression='gzip', dtype={'mcc': str})
            print(f'  {kind} {label}: из сохранённого, строк {len(part):,}')
        else:
            m_start, m_end = month_bounds(label)
            part, last_exc = None, None
            for attempt in range(1, FETCH_RETRIES + 1):
                mem = MEM_LIMIT_FALLBACKS[min(attempt - 1, len(MEM_LIMIT_FALLBACKS) - 1)]
                print(f'  {kind} {label}: Impala, mem={mem}, попытка {attempt}/{FETCH_RETRIES}...', flush=True)
                try:
                    with imp:
                        try:
                            imp.execute(f'set MEM_LIMIT={mem}')
                        except Exception:
                            pass
                        part = imp.fetch(sql_builder(m_start, m_end))
                    break
                except Exception as exc:
                    last_exc = exc
                    print(f'  {kind} {label}: ошибка {type(exc).__name__}: {str(exc)[:300]}')
                    if not (_is_admission_fail(exc) and attempt < FETCH_RETRIES):
                        break
            if part is None:
                raise RuntimeError(f'{kind} {label}: не удалось получить из Impala') from last_exc
            if len(part) == 0:
                raise RuntimeError(f'{kind} {label}: Impala вернула 0 строк — месяц не грузим')
            part = part.copy()
            part.to_csv(path, index=False, compression='gzip')
            print(f'  {kind} {label}: строк {len(part):,}, сохранено {path.name}')
        part['report_month'] = label
        parts.append(part)
    raw = pd.concat(parts, ignore_index=True)
    raw['mcc'] = raw['mcc'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    for c in ['trx_cnt', 'trx_sum', 'commission_from_ops']:
        raw[c] = pd.to_numeric(raw[c], errors='coerce').fillna(0.0)
    return raw

## 2) MCC по месяцам

Колонки как в `mcc_month_upload_jan_aug.ipynb`: доли считаются внутри месяца, поэтому месяцы можно догружать по отдельности.

In [ ]:
mcc_raw = fetch_months('mcc', sql_mcc_month)

g = (
    mcc_raw.groupby(['report_month', 'mcc'], as_index=False)
    .agg(trx_cnt=('trx_cnt', 'sum'), trx_sum=('trx_sum', 'sum'), commission_from_ops=('commission_from_ops', 'sum'))
)
g['snapshot_month_start'] = g['report_month'] + '-01'
month_trx = g.groupby('report_month')['trx_sum'].transform('sum')
month_cnt = g.groupby('report_month')['trx_cnt'].transform('sum')
g['acq_pct'] = np.where(g['trx_sum'] == 0, np.nan, g['commission_from_ops'] / g['trx_sum'] * 100.0)
g['share_trx_sum_pct'] = np.where(month_trx == 0, np.nan, g['trx_sum'] / month_trx * 100.0)
g['share_trx_cnt_pct'] = np.where(month_cnt == 0, np.nan, g['trx_cnt'] / month_cnt * 100.0)
mcc_month_df = g[[
    'report_month', 'snapshot_month_start', 'mcc', 'trx_cnt', 'trx_sum', 'commission_from_ops',
    'acq_pct', 'share_trx_sum_pct', 'share_trx_cnt_pct',
]].sort_values(['report_month', 'trx_sum'], ascending=[True, False]).reset_index(drop=True)

print('Норма: ~135–140 MCC в месяц, сумма долей ~100')
display(mcc_month_df.groupby('report_month').agg(
    mcc_n=('mcc', 'nunique'), trx_cnt=('trx_cnt', 'sum'), trx_sum=('trx_sum', 'sum'), share_sum=('share_trx_sum_pct', 'sum')
))

## 3) Типы карт по месяцам

Классификация платёжной системы — как в `card_type_month_upload_jan_aug.ipynb`.

In [ ]:
EMPTY_LABELS = {'none', 'nan', 'null', 'unknown', 'empty'}


def _norm_desc(text):
    return '' if pd.isna(text) else str(text).strip().lower()


def classify_scheme_raw(text):
    s = _norm_desc(text)
    if not s or s in EMPTY_LABELS:
        return None
    if re.search(r'union\s*pay|unionpay|юнион', s):
        return 'UnionPay'
    if re.search(r'master\s*card|mastercard|\bmaestro\b', s):
        return 'Mastercard'
    if re.search(r'\bvisa\b|виза', s):
        return 'Visa'
    if re.search(r'(^|[^a-zа-я])мир([^a-zа-я]|$)|nspk\s*mir|\bmir\b', s):
        return 'МИР'
    if re.search(r'\bjcb\b', s):
        return 'JCB'
    if re.search(r'amex|american\s*express', s):
        return 'Amex'
    if re.search(r'\bsbp\b|сбп', s):
        return 'СБП'
    return None


def is_rshb_issuer(text):
    s = _norm_desc(text)
    if not s or s in EMPTY_LABELS:
        return False
    if re.search(r'рсхб|rshb|россельхоз', s):
        return True
    if re.search(r'\bрф\b|российский\s*филиал', s) or s.endswith('рф'):
        return True
    if re.search(r'головн\w*\s+офис|\bцрмб\b', s):
        return True
    return False


def classify_payment_system(text):
    s = _norm_desc(text)
    if not s or s in EMPTY_LABELS:
        return 'прочие'
    raw = classify_scheme_raw(text)
    if raw in ('UnionPay', 'Mastercard', 'Visa', 'МИР'):
        return raw
    if raw in ('JCB', 'Amex', 'СБП'):
        return 'прочие'
    if is_rshb_issuer(text):
        return 'Карта РСХБ'
    return 'прочие'


card_raw = fetch_months('card_type', sql_card_type_month)
card_raw['payment_system'] = card_raw['card_type'].map(classify_payment_system)
card_layer = (
    card_raw.groupby(['report_month', 'mcc', 'payment_system'], dropna=False)
    .agg(trx_cnt=('trx_cnt', 'sum'), trx_sum=('trx_sum', 'sum'), commission_from_ops=('commission_from_ops', 'sum'))
    .reset_index()
)[['report_month', 'mcc', 'payment_system', 'trx_cnt', 'trx_sum', 'commission_from_ops']]

print('Норма: ~530–560 строк в месяц, как в январе–августе')
display(card_layer.groupby('report_month').agg(rows=('mcc', 'size'), trx_cnt=('trx_cnt', 'sum'), trx_sum=('trx_sum', 'sum')))
display(card_layer.pivot_table(index='payment_system', columns='report_month', values='trx_sum', aggfunc='sum').round(0))

## 4) Запись в DRP: удалить эти месяцы и дописать

Колонки берутся в порядке таблицы в DRP. Если в расчёте нет какой-то колонки таблицы — ячейка остановится до удаления.

In [ ]:
MONTH_SQL = 'SUBSTRING(BTRIM(CAST(report_month AS TEXT)) FROM 1 FOR 7)'

if 'drp' not in globals() or drp is None:
    drp = connect(
        to='DRP',
        user_params={'user_name': input('DRP user: ').strip(), 'password': getpass.getpass('DRP password: ')},
    )


def to_text_df(df):
    out = df.copy()
    out.columns = [str(c).strip().lower() for c in out.columns]
    for c in out.columns:
        out[c] = out[c].map(lambda x: None if pd.isna(x) else str(x)).astype(object)
    return out


def replace_months(table, df):
    fq = f'{DRP_SCHEMA}.{table}'
    up = to_text_df(df)
    months = sorted(up['report_month'].str[:7].unique().tolist())
    assert months == MONTHS, f'{table}: в расчёте месяцы {months}, ожидались {MONTHS}'
    by_month = f'SELECT {MONTH_SQL} AS m, COUNT(*) AS n FROM {fq} GROUP BY 1 ORDER BY 1'
    months_in = ', '.join(f"'{m}'" for m in months)
    with drp:
        db_cols = drp.fetch(
            'SELECT column_name FROM information_schema.columns '
            f"WHERE table_schema = '{DRP_SCHEMA}' AND table_name = '{table}' ORDER BY ordinal_position"
        )['column_name'].tolist()
        missing = [c for c in db_cols if c not in up.columns]
        if not db_cols or missing:
            raise RuntimeError(f'{fq}: нет таблицы или в расчёте нет колонок {missing}. Ничего не удалено.')
        before = drp.fetch(by_month)
        drp.execute(f'DELETE FROM {fq} WHERE {MONTH_SQL} IN ({months_in})')
        drp.write(table=fq, df=up[db_cols], mode='append')
        after = drp.fetch(by_month)
    res = before.merge(after, on='m', how='outer', suffixes=('_before', '_after')).sort_values('m')
    print(f'=== {fq}: записано строк {len(up):,} за {months} ===')
    display(res)
    old = res[~res['m'].isin(months)]
    if not (old['n_before'].astype(float) == old['n_after'].astype(float)).all():
        print('ВНИМАНИЕ: изменились строки других месяцев, пришлите скриншот')
    return res


res_mcc = replace_months(MCC_TABLE, mcc_month_df)
res_card = replace_months(CARD_TABLE, card_layer)
print('ГОТОВО. Дашборд: Ctrl+F5 и ⋮ → Обновить дашборд, выбрать 2026-09.')